# 07 - Results Interpretation

**Purpose:** consolidate the final results around the project question:

> Can time-series history, social-network exposure, and review-language signals help forecast short-term shifts in community attention toward local Yelp businesses?

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

METRICS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_metrics.csv"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_predictions.csv"
PULSE_METRICS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_metrics.csv"
PULSE_PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_predictions.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
FEATURE_SUMMARY_PATH = PROCESSED_DIR / "forecasting_feature_summary.json"

metrics = pd.read_csv(METRICS_OUTPUT_PATH)
predictions = pd.read_csv(PREDICTIONS_OUTPUT_PATH)
pulse_metrics = pd.read_csv(PULSE_METRICS_OUTPUT_PATH)
pulse_predictions = pd.read_csv(PULSE_PREDICTIONS_OUTPUT_PATH)
with GRAPH_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    graph_summary = json.load(file)
with FEATURE_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

metrics.sort_values(["split", "WAPE", "MAE"])

,split,task,model,rows,MAE,RMSE,WAPE
0,primary_covid_test,review_count_regression,Baseline: last month,21024,1.601170,3.007150,0.664292
1,primary_covid_test,review_count_regression,Baseline: rolling 3-month avg,21024,1.646071,3.364436,0.682921
2,primary_covid_test,review_count_regression,ML: historical,21024,1.791595,3.432584,0.743295
3,primary_covid_test,review_count_regression,ML: historical + SNA,21024,1.797421,3.397139,0.745712
4,primary_covid_test,review_count_regression,ML: historical + NLP,21024,1.798615,3.426518,0.746208
5,primary_covid_test,review_count_regression,ML: historical + business,21024,1.839525,3.513199,0.763181
6,primary_covid_test,review_count_regression,ML: all modalities,21024,1.871549,3.507464,0.776467
7,primary_covid_test,review_count_regression,Baseline: seasonal naive,21024,3.596128,7.490338,1.491959
8,secondary_pre_covid_test,review_count_regression,ML: historical + business,10511,2.230412,3.532347,0.369195
9,secondary_pre_covid_test,review_count_regression,ML: all modalities,10511,2.238087,3.538540,0.370465


## Regression Interpretation

Summarize which feature group performs best in each time split and whether the all-modality model improves on simpler alternatives.


In [2]:
regression_summary_rows = []
for split_name, split_metrics in metrics.groupby("split"):
    ranked = split_metrics.sort_values("WAPE").reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    regression_summary_rows.append({
        "split": split_name,
        "best_model": best["model"],
        "best_WAPE": best["WAPE"],
        "historical_WAPE": hist["WAPE"],
        "business_WAPE": business["WAPE"],
        "sna_WAPE": sna["WAPE"],
        "nlp_WAPE": nlp["WAPE"],
        "all_modalities_WAPE": all_modalities["WAPE"],
        "all_vs_historical_relative_change": (all_modalities["WAPE"] - hist["WAPE"]) / hist["WAPE"],
        "all_vs_business_relative_change": (all_modalities["WAPE"] - business["WAPE"]) / business["WAPE"],
    })
regression_summary = pd.DataFrame(regression_summary_rows)
regression_summary

,split,best_model,best_WAPE,historical_WAPE,business_WAPE,sna_WAPE,nlp_WAPE,all_modalities_WAPE,all_vs_historical_relative_change,all_vs_business_relative_change
0,primary_covid_test,Baseline: last month,0.664292,0.743295,0.763181,0.745712,0.746208,0.776467,0.044627,0.017409
1,secondary_pre_covid_test,ML: historical + business,0.369195,0.380410,0.369195,0.378079,0.380267,0.370465,-0.026142,0.003441


## Pulse Interpretation

Summarize pulse-classification performance with class balance in mind. The goal is to understand whether models can rank or detect unusual attention shifts.


In [3]:
# Pulse summaries emphasize rare-event detection rather than overall accuracy.
pulse_summary_rows = []
for split_name, split_metrics in pulse_metrics.groupby("split"):
    ranked = split_metrics.sort_values(["F1", "PR_AUC"], ascending=[False, False]).reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    pulse_summary_rows.append({
        "split": split_name,
        "positive_rate": all_modalities["positive_rate"],
        "best_model": best["model"],
        "best_F1": best["F1"],
        "best_PR_AUC": best["PR_AUC"],
        "historical_F1": hist["F1"],
        "business_F1": business["F1"],
        "sna_F1": sna["F1"],
        "nlp_F1": nlp["F1"],
        "all_modalities_F1": all_modalities["F1"],
        "all_modalities_PR_AUC": all_modalities["PR_AUC"],
    })
pulse_summary = pd.DataFrame(pulse_summary_rows)
pulse_summary

,split,positive_rate,best_model,best_F1,best_PR_AUC,historical_F1,business_F1,sna_F1,nlp_F1,all_modalities_F1,all_modalities_PR_AUC
0,primary_covid_test,0.102407,Baseline: rising recent activity,0.280882,0.148129,0.167985,0.183338,0.139422,0.136761,0.164012,0.176227
1,secondary_pre_covid_test,0.131957,ML: historical,0.300332,0.244637,0.300332,0.285534,0.277992,0.293703,0.273235,0.232458


In [4]:
print("Social graph summary")
for key, value in graph_summary.items():
    print(f"{key}: {value}")

print("\nForecasting dataset summary")
for key, value in feature_summary.items():
    print(f"{key}: {value}")

Social graph summary
active_review_threshold: 5
threshold_candidates: [2, 3, 5, 10, 20]
edge_weight_formula: 1 + log1p(shared_business_count) + category_jaccard
reviewing_users: 245421
matched_user_profiles: 245419
active_users: 26598
graph_nodes: 26598
graph_edges: 116558
mean_edge_weight: 1.833072733525831
mean_edge_shared_business_count: 2.048550936014688
mean_edge_category_jaccard: 0.2671618532172656
connected_components: 11508
largest_component_size: 14965
isolated_active_users: 11387
community_method: weighted_louvain_largest_component
communities_assigned: 63
threshold_sensitivity_output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\active_reviewer_threshold_sensitivity.csv
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\user_network_features.csv

Forecasting dataset summary
min_total_reviews: 100
min_active_months: 36
business_count: 876
row_count: 68249
feature_month_min: 2015-01
feature

In [5]:
for _, row in regression_summary.iterrows():
    split = row["split"]
    all_vs_hist = row["all_vs_historical_relative_change"] * 100
    all_vs_business = row["all_vs_business_relative_change"] * 100
    print(f"{split} regression:")
    print(f"  Best model: {row['best_model']} with WAPE={row['best_WAPE']:.4f}")
    print(f"  All modalities WAPE: {row['all_modalities_WAPE']:.4f}")
    print(f"  All modalities vs historical: {all_vs_hist:+.2f}%")
    print(f"  All modalities vs historical+business: {all_vs_business:+.2f}%")

print()
for _, row in pulse_summary.iterrows():
    split = row["split"]
    print(f"{split} attention pulses:")
    print(f"  Positive rate: {row['positive_rate']:.3f}")
    print(f"  Best model: {row['best_model']} with F1={row['best_F1']:.4f}, PR-AUC={row['best_PR_AUC']:.4f}")
    print(f"  All modalities F1: {row['all_modalities_F1']:.4f}, PR-AUC={row['all_modalities_PR_AUC']:.4f}")

primary_covid_test regression:
  Best model: Baseline: last month with WAPE=0.6643
  All modalities WAPE: 0.7765
  All modalities vs historical: +4.46%
  All modalities vs historical+business: +1.74%
secondary_pre_covid_test regression:
  Best model: ML: historical + business with WAPE=0.3692
  All modalities WAPE: 0.3705
  All modalities vs historical: -2.61%
  All modalities vs historical+business: +0.34%

primary_covid_test attention pulses:
  Positive rate: 0.102
  Best model: Baseline: rising recent activity with F1=0.2809, PR-AUC=0.1481
  All modalities F1: 0.1640, PR-AUC=0.1762
secondary_pre_covid_test attention pulses:
  Positive rate: 0.132
  Best model: ML: historical with F1=0.3003, PR-AUC=0.2446
  All modalities F1: 0.2732, PR-AUC=0.2325


## Interpretation Structure

1. **Time series:** recent review patterns are the strongest reference point, especially during the COVID-era test where the last-month baseline is hardest to beat.
2. **Business metadata:** static context helps most in the pre-COVID split, where the best review-count model is historical + business.
3. **SNA:** social exposure is measurable and reportable, but it does not consistently improve prediction after temporal and business signals are included.
4. **NLP:** lightweight review-language features add an extra modality, but their incremental gain is modest.
5. **Target design:** attention pulses are more aligned with community-attention shifts than raw volume alone, while remaining difficult because pulse events are relatively rare.


## Final Position

The main empirical result is mixed but useful: simple temporal baselines remain very strong under COVID-era disruption, while ML models help more in the pre-COVID split. SNA and NLP are valuable for interpretation and modality coverage, but they provide limited incremental predictive lift in this dataset.

Key limitations:

- Yelp friendship links are static.
- SNA features measure exposure, not causal influence.
- NLP features are lightweight lexicon/text-length signals.
- COVID-era disruption changes predictability.
- Review activity is a proxy for Yelp attention, not revenue or true customer volume.
- Static business metadata may include end-of-dataset information.